In [ ]:
import pandas as pd   
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:

df_raw = pd.read_csv(r"data/loan_approval_data.csv")
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.isnull().sum()

In [ ]:
df_raw.nunique()

In [ ]:
df_raw.shape

In [ ]:
df_raw.describe()

In [ ]:
#  handling the missing values 
# for catagorial
categorical_cols = df_raw.select_dtypes(include=["object"]).columns
numerical_col = df_raw.select_dtypes(include=["float64"]).columns


In [ ]:
categorical_cols

In [ ]:
numerical_col

In [ ]:
categorical_cols.size + numerical_col.size

In [ ]:
#  creating simple imputer in here 

num_imp = SimpleImputer(strategy="mean")
cat_imp = SimpleImputer(strategy="most_frequent")

In [ ]:
df_raw[numerical_col] = num_imp.fit_transform(df_raw[numerical_col])
df_raw[categorical_cols] = cat_imp.fit_transform(df_raw[categorical_cols])

In [ ]:
df_raw.isnull().sum()

In [ ]:
df_raw.head()

In [ ]:
#   we will start with the EDA 
#  exploratery data anaylsis

#  first checking how balance our classes are (here we are checking how distrubuted out target value is )

classes_count = df_raw["Loan_Approved"].value_counts()

#  small pir chart to get the idea
plt.pie(classes_count, labels=["No", "Yes"], autopct="%1.1f")
plt.title("is loan approved or no")

In [ ]:
# analyzing the catagory

gender_cnt = df_raw["Gender"].value_counts()
ax=sns.barplot(gender_cnt)
ax.bar_label(ax.containers[0])

edu_cnt = df_raw["Education_Level"].value_counts()
ax=sns.barplot(edu_cnt)
ax.bar_label(ax.containers[-1])

In [ ]:
# analysizing the income

sns.histplot(
    data = df_raw,
    x = "Applicant_Income",
    bins=20
)

In [ ]:
sns.histplot(
    data = df_raw,
    x = "Coapplicant_Income",
    bins=30
)

In [ ]:
#  for the outliers- box plot

sns.boxplot(
    data=df_raw,
    x="Loan_Approved",
    y="Applicant_Income"
)

In [ ]:
fig, axes = plt.subplots(2, 2)

sns.boxplot(ax=axes[0,0], data=df_raw, x="Loan_Approved", y="Applicant_Income")
sns.boxplot(ax=axes[0,1], data=df_raw, x="Loan_Approved", y="Credit_Score")
sns.boxplot(ax=axes[1,0], data=df_raw, x="Loan_Approved", y="DTI_Ratio")
sns.boxplot(ax=axes[1,1], data=df_raw, x="Loan_Approved", y="Savings")

plt.tight_layout()

In [ ]:
#  what is the relation with the credit score with theapproved loan 

sns.histplot(
    data=df_raw,
    x="Credit_Score",
    hue="Loan_Approved",
    bins=20,
    multiple="dodge"
)


In [ ]:

sns.histplot(
    data=df_raw,
    x="Applicant_Income",
    hue="Loan_Approved",
    bins=20,
    multiple="dodge"
)


In [ ]:
#  remove the applicant id

df_raw.drop(columns="Applicant_ID")

In [ ]:
#  one hot encoading, kinda the feature engineering
df_raw.head()

In [ ]:
# encoding

# Label_encoder = assigns the values to the binary things (gender)
# (used for ordinal data)
# where we have tje level for ythe undergraduate and the graduate 

# one hot encoder = does the same things as the pandas get dummies 
# used fro the nominal data 
# here where we dont care for the data

df_raw.info()


In [ ]:
cols = ["Employment_Status", "Marital_Status", "Gender", "Loan_Purpose", "Property_Area", "Employer_Category"]

In [ ]:
#  this is for the ordinal data
le = LabelEncoder()
df_raw["Loan_Approved"] = le.fit_transform(df_raw["Loan_Approved"])
df_raw["Education_Level"] = le.fit_transform(df_raw["Education_Level"])

In [ ]:
df_raw.head()

In [ ]:
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded = ohe.fit_transform(df_raw[cols])

In [ ]:
encoded

In [ ]:
encoded_df=pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=df_raw.index)

In [ ]:
encoded_df.head()

In [ ]:
df_process = pd.concat([df_raw.drop(columns=cols), encoded_df], axis=1)

In [ ]:
df_process.head()

In [ ]:
#  lets dp the correlation heatmap

name_col = df_process.select_dtypes(include=['number'])
corr_matrix = name_col.corr()

In [ ]:
name_col.corr()["Loan_Approved"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(15,8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt = ".2f",
    cmap="coolwarm"
)

In [ ]:
#  for the above chart if 1 than we have strong corelation if -1 we have recopical of the relation and 0 means there is no linear correlation 

In [ ]:
#  train test split 
X = df_process.drop(columns=["Loan_Approved"])
y = df_process["Loan_Approved"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [ ]:
y.value_counts(str
               )

In [ ]:
y_test.value_counts(normalize=True)

In [ ]:
y_train.value_counts(normalize=True)

In [ ]:
#  train and evaluate the model]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
#  logistics regression 

from sklearn.linear_model import LogisticRegression

model_logistics = LogisticRegression(max_iter=1000)
model_logistics.fit(X_train_scaled, y_train)
y_pred = model_logistics.predict(X_test_scaled)


In [ ]:
# evulation
from sklearn.metrics import  accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Logistics regression model")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
    

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)
y_pred = knn_model.predict(X_test_scaled)

In [ ]:
print("Knn model ")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB

model_naive_bayes = GaussianNB()
model_naive_bayes.fit(X_train_scaled, y_train)
y_pred = model_naive_bayes.predict(X_test_scaled)

In [ ]:
print("Naive Bayes model ")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
df_process.head()

In [ ]:
df_raw["Applicant_Income"].head()

In [ ]:
# Restore original income values
df_process["Applicant_Income"] = df_raw.loc[
    df_process.index, "Applicant_Income"
]

# Recreate logarithmic feature correctly
df_process["Applicant_Income_log"] = np.log1p(
    df_process["Applicant_Income"]
)

In [ ]:
import numpy as np

print(df_process[[
    "Applicant_Income",
    "Applicant_Income_log"
]].head())

print(
    np.isinf(
        df_process.select_dtypes(include="number")
    ).sum()
)

In [ ]:
df_raw = pd.read_csv("data/loan_approval_data.csv")

In [ ]:
df_process["Applicant_Income_log"] = np.log1p(
    df_process["Applicant_Income"]
)

In [ ]:
df_process["DTI_Ratio_sq"] = df_process["DTI_Ratio"]**2
df_process["Credit_Score_sq"] = df_process["Credit_Score"]**2

In [ ]:
X = df_process.drop(columns=["Loan_Approved", "Credit_Score", "DTI_Ratio"])
y = df_process["Loan_Approved"]

In [ ]:
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression

model_logistics = LogisticRegression(max_iter=1000)
model_logistics.fit(X_train_scaled, y_train)
y_pred = model_logistics.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import  accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Logistics regression model")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)
y_pred = knn_model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import  accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("KNN model")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB

model_naive_bayes = GaussianNB()
model_naive_bayes.fit(X_train_scaled, y_train)
y_pred = model_naive_bayes.predict(X_test_scaled)

In [ ]:
print("Naive Bayes model ")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))